# Compose an end-to-end policy investigation

This bounded, deterministic, CPU-only tutorial follows one frozen TorchRL policy through a complete evidence chain:

1. evaluate competence on every non-terminal state in a corridor;
2. identify a hidden channel associated with the goal direction;
3. trace that channel's relevance back to the observation;
4. intervene on the selected channel and matched channel controls;
5. measure open-loop decisions and closed-loop behavior.

The example adapts the useful corridor fixture from [TDHook PR #105](https://github.com/Xmaster6y/tdhook/pull/105), while execution uses public XDRL typed-interaction and paired-workflow/provenance APIs. The result applies only to the synthetic fitted-Q policy trained below; it is not a paper reproduction or evidence about navigation policies in general.

In [ ]:
import torch
from tensordict import TensorDict
from tensordict.nn import TensorDictModule
from torch import nn
from tdhook.attribution import LRP
from tdhook.concepts import ChannelConditionedLRP, ConceptSelection
from tdhook.latent import ActivationCaching, SteeringVectors
from tdhook.targets import Target
from tdhook.workflow import Workflow
from xdrl import (
    ArtifactDigestAlgorithm,
    BatchSemantics,
    InputArtifactReference,
    InputArtifactRole,
    InteractionContract,
    InteractionPhase,
    KeyPresence,
    KeyRole,
    KeySchema,
    ModelRole,
    module_digest,
    named_tensor_digest,
    OutputArtifactDeclaration,
    OutputArtifactDigest,
    OutputArtifactRole,
    RuntimeInteractionContext,
    TDHookWorkflowPairManifest,
    TDHookWorkflowRunner,
    TensorDictSchema,
    WorkflowProvenance,
)

torch.manual_seed(7)
torch.set_num_threads(1)
CORRIDOR_LENGTH = 9
FEATURES = 8
SEED = 7
CODE_REVISION = "tutorial:issue-68-v1"
DEVICE = torch.device("cpu")

## 1. Freeze a complete decision panel

An agent and goal occupy a nine-cell corridor. The two observation planes encode their positions; actions are `0 = left` and `1 = right`. Enumerating every non-terminal state and both transitions makes the evaluation set fixed rather than sampled.

In [ ]:
def encode_state(agent_position, goal_position):
    observation = torch.zeros(2, CORRIDOR_LENGTH, device=DEVICE)
    observation[0, agent_position] = 1.0
    observation[1, goal_position] = 1.0
    return observation


observations = []
next_observations = [[], []]
rewards = [[], []]
dones = [[], []]
optimal_actions = []
goal_is_right = []
state_pairs = []
for agent_position in range(CORRIDOR_LENGTH):
    for goal_position in range(CORRIDOR_LENGTH):
        if agent_position == goal_position:
            continue
        observations.append(encode_state(agent_position, goal_position))
        optimal_actions.append(int(goal_position > agent_position))
        goal_is_right.append(int(goal_position > agent_position))
        state_pairs.append((agent_position, goal_position))
        for action, displacement in ((0, -1), (1, 1)):
            next_position = max(0, min(CORRIDOR_LENGTH - 1, agent_position + displacement))
            done = next_position == goal_position
            next_observations[action].append(encode_state(next_position, goal_position))
            rewards[action].append(1.0 if done else -0.02)
            dones[action].append(done)

observations = torch.stack(observations)
next_observations = torch.stack([torch.stack(values) for values in next_observations], dim=1)
rewards = torch.tensor(rewards, device=DEVICE).T
dones = torch.tensor(dones, device=DEVICE).T
optimal_actions = torch.tensor(optimal_actions, device=DEVICE)
goal_is_right = torch.tensor(goal_is_right, device=DEVICE)
evaluation_panel = TensorDict(
    {"observation": observations, "target_action": optimal_actions, "concept_labels": goal_is_right},
    batch_size=[len(observations)],
)
{"states": len(observations), "left_right_balance": torch.bincount(goal_is_right).tolist()}

## 2. Fit, freeze, and type the policy

Fitted Q-iteration uses the complete transition table. After training, a public TorchRL `TensorDictModule` and an XDRL `InteractionContract` declare the evaluation boundary. The checkpoint identity is the SHA-256 digest of the frozen parameters.

In [ ]:
class CorridorQNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.features = nn.Sequential(
            nn.Linear(2 * CORRIDOR_LENGTH, 16), nn.ReLU(), nn.Linear(16, FEATURES), nn.ReLU()
        )
        self.policy_head = nn.Linear(FEATURES, 2)

    def forward(self, observation):
        return self.policy_head(self.features(self.flatten(observation)))


q_network = CorridorQNetwork().to(DEVICE)
target_network = CorridorQNetwork().to(DEVICE)
target_network.load_state_dict(q_network.state_dict())
optimizer = torch.optim.Adam(q_network.parameters(), lr=0.02)
for iteration in range(600):
    with torch.no_grad():
        next_values = target_network(next_observations.flatten(0, 1)).max(-1).values.reshape(len(observations), 2)
        bellman_targets = rewards + 0.95 * (~dones) * next_values
    loss = nn.functional.smooth_l1_loss(q_network(observations), bellman_targets)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if iteration % 10 == 0:
        target_network.load_state_dict(q_network.state_dict())

q_network.eval()


class GreedyCorridorPolicy(nn.Module):
    def __init__(self, q_network):
        super().__init__()
        self.q_network = q_network

    def forward(self, observation):
        action_value = self.q_network(observation)
        return action_value, action_value.argmax(-1)


policy = TensorDictModule(
    GreedyCorridorPolicy(q_network),
    in_keys=["observation"],
    out_keys=["action_value", "action"],
)
batch_semantics = BatchSemantics(("state",))
input_schema = TensorDictSchema(
    (KeySchema("observation", KeyRole.OBSERVATION, KeyPresence.REQUIRED),), batch_semantics
)
output_schema = TensorDictSchema(
    (
        KeySchema("action_value", KeyRole.VALUE, KeyPresence.PRODUCED),
        KeySchema("action", KeyRole.ACTION, KeyPresence.PRODUCED),
    ),
    batch_semantics,
)


checkpoint_sha256 = module_digest(q_network)
contract = InteractionContract(
    identity="corridor-policy:evaluation:v1",
    role=ModelRole.ACTOR,
    phase=InteractionPhase.EVALUATION,
    module_path="policy",
    input_schema=input_schema,
    output_schema=output_schema,
    model_id="corridor-fitted-q-8-channel",
    checkpoint_id=f"sha256:{checkpoint_sha256}",
    module_training=False,
)
interaction = RuntimeInteractionContext(contract, policy, evaluation_panel)
baseline_output = interaction(evaluation_panel.select("observation").clone())
baseline_actions = baseline_output["action"]
policy_accuracy = float((baseline_actions == optimal_actions).float().mean())
assert policy_accuracy == 1.0
{"checkpoint_sha256": checkpoint_sha256, "complete_panel_accuracy": policy_accuracy}

In [ ]:
state_index = {pair: index for index, pair in enumerate(state_pairs)}


def rollout(action_table, start, goal, max_steps=12):
    position = start
    path = [position]
    for _ in range(max_steps):
        if position == goal:
            break
        action = int(action_table[state_index[(position, goal)]])
        position = max(0, min(CORRIDOR_LENGTH - 1, position + (-1 if action == 0 else 1)))
        path.append(position)
    return path, position == goal


baseline_rollouts = [rollout(baseline_actions, start, goal) for start, goal in state_pairs]
baseline_success = sum(success for _path, success in baseline_rollouts) / len(baseline_rollouts)
assert baseline_success == 1.0
{
    "closed_loop_success": baseline_success,
    "left_goal_example": rollout(baseline_actions, 6, 1)[0],
    "right_goal_example": rollout(baseline_actions, 2, 7)[0],
}

## 3. Run concept-conditioned attribution through XDRL

The concept label says whether the goal lies to the right. LRP first attributes each chosen action value to the hidden layer. `ConceptSelection` then ranks channels by the contrast between right- and left-goal states, and channel-conditioned LRP traces the selected channel back to the input. This is an association and attribution result, not yet a causal claim.

The diagnostic contract explicitly enables gradients. XDRL checks that contract against TDHook's public workflow plan and records actual model-call lifecycle evidence.

In [ ]:
def chosen_action_value(targets, additional):
    values = targets["action_value"]
    chosen = values.gather(-1, additional["target_action"].unsqueeze(-1)).squeeze(-1)
    return TensorDict({"chosen_action_value": chosen}, batch_size=targets.batch_size)


lrp_options = {
    "init_attr_targets": chosen_action_value,
    "additional_init_keys": ["target_action"],
    "warn_on_missing_rule": False,
}
diagnosis = Workflow(
    LRP(
        input_modules=["module.q_network.features.2"],
        attribution_key=("attributions", "concept_examples"),
        **lrp_options,
    ),
    ConceptSelection(("attributions", "concept_examples", "module.q_network.features.2"), direction="negative"),
    ChannelConditionedLRP(LRP(**lrp_options), condition_module="module.q_network.features.2"),
)
diagnostic_contract = InteractionContract(
    identity="corridor-policy:diagnosis:v1",
    role=ModelRole.ACTOR,
    phase=InteractionPhase.EVALUATION,
    module_path="policy",
    input_schema=input_schema,
    output_schema=output_schema,
    model_id=contract.model_id,
    checkpoint_id=contract.checkpoint_id,
    module_training=False,
    gradient_enabled=True,
)
checkpoint_artifact = InputArtifactReference(
    "checkpoint:corridor-fitted-q-v1",
    InputArtifactRole.MODEL_CHECKPOINT,
    ArtifactDigestAlgorithm.SHA256,
    checkpoint_sha256,
    metadata={"synthetic": True, "training_seed": SEED, "training_iterations": 600},
)
diagnostic_runner = TDHookWorkflowRunner(RuntimeInteractionContext(diagnostic_contract, policy, evaluation_panel))
diagnostic_plan = diagnostic_runner.plan(diagnosis, evaluation_panel)
diagnostic_execution = diagnostic_runner.run(
    diagnosis,
    evaluation_panel.clone(),
    code_revision=CODE_REVISION,
    expected_plan=diagnostic_plan,
    seed=SEED,
    input_artifacts=(checkpoint_artifact,),
    callback_identifiers={chosen_action_value: "tutorial.chosen-action-value/v1"},
)
selection = diagnostic_execution.data["metrics", "concept_selection"]
selected_channel = int(selection["channel"][0])
conditioned_relevance = diagnostic_execution.data["attributions", "conditioned", "observation"]
assert selected_channel == 0
assert diagnostic_execution.provenance.model_calls == diagnostic_plan.model_passes == 2
assert WorkflowProvenance.from_json(diagnostic_execution.provenance.to_json()) == diagnostic_execution.provenance
{
    "selected_left_associated_channel": selected_channel,
    "selection_score": float(selection["score"][0]),
    "conditioned_input_relevance_shape": tuple(conditioned_relevance.shape),
    "planned_and_observed_model_calls": diagnostic_execution.provenance.model_calls,
}

## 4. Intervene with matched controls and provenance

We first cache the complete hidden representation, then replace one channel with its mean activation in right-goal states. Every channel receives the same type of intervention and evaluation panel; non-selected channels are specificity controls.

Each comparison uses `TDHookWorkflowRunner.run_paired`. The baseline is an explicit no-op at the same target. XDRL verifies the declared workflow difference, restores model and RNG state between arms, binds input/output artifact digests, and emits a tensor-free manifest. That manifest establishes matched mechanics and provenance; the behavioral contrast supplies the empirical result.

In [ ]:
runner = TDHookWorkflowRunner(interaction)
capture = runner.run(
    Workflow(ActivationCaching("module.q_network.features.2", cache_key=("activations", "candidate_layer"))),
    evaluation_panel.select("observation").clone(),
    code_revision=CODE_REVISION,
    seed=SEED,
    input_artifacts=(checkpoint_artifact,),
)
feature_values = capture.data["activations", "candidate_layer", "module.q_network.features.2"].detach()
right_context_means = feature_values[goal_is_right == 1].mean(0)


def no_op(*, output, **_):
    return output


def replacement_callback(value):
    def replace(*, output, **_):
        return torch.full_like(output, value)

    return replace


def result_resolver(data, declarations):
    digest = named_tensor_digest((key, data[key]) for key in ("action", "action_value"))
    return tuple(OutputArtifactDigest(item.identity, ArtifactDigestAlgorithm.SHA256, digest) for item in declarations)


def run_channel_pair(channel):
    target = Target("module.q_network.features.2", "activation", -1, (channel,))
    replace = replacement_callback(float(right_context_means[channel]))
    pair = runner.run_paired(
        Workflow(SteeringVectors([target], steer_fn=no_op)),
        Workflow(SteeringVectors([target], steer_fn=replace)),
        evaluation_panel.select("observation"),
        pair_id=f"corridor-policy:channel-{channel}:right-context",
        code_revision=CODE_REVISION,
        declared_workflow_differences=(0,),
        seed=SEED,
        input_artifacts=(checkpoint_artifact,),
        baseline_output_artifacts=(
            OutputArtifactDeclaration(f"result:channel-{channel}:baseline", OutputArtifactRole.INTERVENTION_RESULT),
        ),
        baseline_output_artifact_resolver=result_resolver,
        intervention_output_artifacts=(
            OutputArtifactDeclaration(
                f"result:channel-{channel}:intervention", OutputArtifactRole.INTERVENTION_RESULT
            ),
        ),
        intervention_output_artifact_resolver=result_resolver,
        callback_identifiers={
            no_op: "tutorial.no-op/v1",
            replace: f"tutorial.replace-channel-{channel}-with-right-context-mean/v1",
        },
    )
    assert pair.manifest.interpretation == "mechanics_and_provenance_only"
    assert TDHookWorkflowPairManifest.from_json(pair.manifest.to_json()) == pair.manifest
    assert pair.baseline.provenance.input_artifacts == pair.intervention.provenance.input_artifacts
    return pair


channel_pairs = {channel: run_channel_pair(channel) for channel in range(FEATURES)}
{channel: pair.manifest.changed_steps[0].index for channel, pair in channel_pairs.items()}

## 5. Evaluate decisions and behavior

An activation change is not yet an RL result. We compare greedy actions over the complete panel, then use those frozen action tables in the corridor transition function. The selected-channel intervention should exceed every matched control and disrupt a left-goal rollout while preserving a right-goal rollout.

In [ ]:
pair_actions = {channel: pair.intervention.data["action"] for channel, pair in channel_pairs.items()}
action_flips = {channel: int((actions != baseline_actions).sum()) for channel, actions in pair_actions.items()}
selected_actions = pair_actions[selected_channel]
selected_flips = action_flips[selected_channel]
max_matched_control_flips = max(flips for channel, flips in action_flips.items() if channel != selected_channel)
left_baseline = rollout(baseline_actions, 6, 1)
left_intervention = rollout(selected_actions, 6, 1)
right_baseline = rollout(baseline_actions, 2, 7)
right_intervention = rollout(selected_actions, 2, 7)
intervention_rollouts = [rollout(selected_actions, start, goal) for start, goal in state_pairs]
intervention_success = sum(success for _path, success in intervention_rollouts) / len(intervention_rollouts)

assert selected_flips > max_matched_control_flips
assert left_baseline[1] and not left_intervention[1]
assert right_baseline[1] and right_intervention[1]
behavioral_results = {
    "action_flips_by_channel": action_flips,
    "selected_channel": selected_channel,
    "selected_channel_flips": selected_flips,
    "largest_matched_control_flips": max_matched_control_flips,
    "closed_loop_success": {"baseline": baseline_success, "intervention": intervention_success},
    "left_goal": {"baseline": left_baseline, "intervention": left_intervention},
    "right_goal": {"baseline": right_baseline, "intervention": right_intervention},
}
behavioral_results

## 6. Results and scope

| Stage | Evidence | Supported interpretation |
|---|---|---|
| Competence | Complete-panel accuracy and all-pairs rollouts | Competence in this finite corridor only |
| Concept-conditioned attribution | Selected channel and conditioned input relevance | Association under the configured LRP rules |
| Matched intervention | Declared workflow differences, identical checkpoint/input artifacts, arm output digests | Matched execution mechanics and provenance |
| Specificity controls | Identical interventions on every other hidden channel | The selected intervention is more behaviorally specific in this policy |
| Behavioral evaluation | Complete-panel action changes and left/right rollouts | A causal effect of this activation replacement on this frozen policy |

Nothing here supports a claim about a paper result, another checkpoint, another environment, or a general navigation mechanism. The reusable contribution is the explicit path from competence to a representational hypothesis, matched intervention, controls, provenance, and behavioral evaluation.